In [70]:
import json
import random
from pathlib import Path
from IPython.display import Markdown, display


# =========================
# CONFIG
# =========================

CLEAN_JSON_PATH = Path(
    "/home/rpinter/art-in-swh/artworks/PROMPT_10e_20260531/clean_artworks_label_prediction.json"
)

SRC_ROOT_DIR = Path(
    "/home/rpinter/links/projects/def-baudry/shared/data/artworks/src"
)


# =========================
# HELPERS
# =========================

def read_json(file_path: Path):
    with file_path.open("r", encoding="utf-8") as f:
        return json.load(f)


def normalize_path(path: str) -> str:
    """
    Convert absolute artwork paths to paths relative to SRC_ROOT_DIR.
    """
    path = str(path)

    marker = "/artworks/src/"
    if marker in path:
        return path.split(marker, 1)[-1].lstrip("/")

    return path.lstrip("/")


def ensure_list(value):
    if value is None:
        return []

    if isinstance(value, list):
        return value

    return [value]


def flatten_labels(item: dict) -> set[str]:
    """
    Convert predicted_labels into one flat set of tags.
    """
    labels = item.get("predicted_labels", {})

    tags = set()
    tags.update(ensure_list(labels.get("entities", [])))
    tags.update(ensure_list(labels.get("interaction", [])))
    tags.update(ensure_list(labels.get("outcome", [])))

    return tags


def print_code_file(file_path, language="javascript"):
    """
    Print a source-code file as a Markdown code block in Jupyter.
    """
    file_path = Path(file_path)
    code = file_path.read_text(encoding="utf-8", errors="ignore")

    display(Markdown(f"```{language}\n{code}\n```"))


def print_sample_code(sample, language="javascript"):
    """
    Print the sample tags/labels and then the source code.
    """
    print("File:")
    print(sample["file_path"])

    # print("\nPredicted labels:")
    # print(json.dumps(sample["predicted_labels"], indent=2, ensure_ascii=False))

    print("\nAll tags:")
    print(sample["all_tags"])

    print("\nSource code:")
    file_path = SRC_ROOT_DIR / sample["file_path"]
    print_code_file(file_path, language=language)


def get_random_src_by_tags(
    tags: list[str],
    k: int = 10,
    exact: bool = False,
    clean_json_path: Path | None = None,
    src_root_dir: Path | None = None,
    seed: int = 42,
) -> list[dict]:
    """
    Find k random source-code files from clean_artworks_label_prediction.json.

    exact=False:
        returns files that contain at least all tags passed.

    exact=True:
        returns files whose full label set is exactly equal to the tags passed.

    Excludes files containing "Daniel Shiffman".
    """
    if clean_json_path is None:
        clean_json_path = CLEAN_JSON_PATH

    if src_root_dir is None:
        src_root_dir = SRC_ROOT_DIR

    data = read_json(clean_json_path)

    target_tags = set(tags)
    matches = []

    for item in data:
        item_tags = flatten_labels(item)

        if exact:
            keep = item_tags == target_tags
        else:
            keep = target_tags.issubset(item_tags)

        if not keep:
            continue

        relative_file_path = normalize_path(item["file_path"])
        src_path = src_root_dir / relative_file_path

        if not src_path.exists():
            continue

        src_code = src_path.read_text(encoding="utf-8", errors="ignore")

        if "Daniel Shiffman" in src_code:
            continue

        matches.append({
            "file_path": relative_file_path,
            "predicted_labels": item.get("predicted_labels", {}),
            "label_combination": item.get("label_combination", ""),
            "all_tags": sorted(item_tags),
        })

    rng = random.Random(seed)
    return rng.sample(matches, min(k, len(matches)))


def show_samples(samples: list[dict]):
    """
    Print sample paths and labels, without printing source code.
    """
    for i, sample in enumerate(samples, start=1):
        print("=" * 100)
        print(f"Sample {i}")
        print(f"File: {sample['file_path']}")
        print("Labels:")
        print(json.dumps(sample["predicted_labels"], indent=2, ensure_ascii=False))


In [74]:
tags = [
    # entities
    # "processed_audio",
    # "processed_image",
    # "processed_text",
    "synthesized_sound",
    # "synthesized_text",
    "synthesized_image",
    "randomness",

    # interaction
    "yes",
    # "no",

    # outcome
    "visual",
    "auditory",
    # "static",
    "time_based",
]

samples = get_random_src_by_tags(
    tags=tags,
    k=100,
    # exact=True,
    exact=False,
)
i = 1
print_sample_code(samples[i])

File:
bored89/redo/Project_Template_C17_Cycle_Race-main/sketch.js

All tags:
['auditory', 'processed_image', 'randomness', 'synthesized_image', 'synthesized_sound', 'time_based', 'visual', 'yes']

Source code:


```javascript
var path,mainCyclist;
var player1,player2,player3;
var pathImg,mainRacerImg1,mainRacerImg2;

var oppPink1Img,oppPink2Img;
var oppYellow1Img,oppYellow2Img;
var oppRed1Img,oppRed2Img;
var gameOverImg,cycleBell;

var pinkCG, yellowCG,redCG; 

var END =0;
var PLAY =1;
var gameState = PLAY;

var distance=0;
var gameOver, restart;

function preload(){
  pathImg = loadImage("images/Road.png");
  mainRacerImg1 = loadAnimation("images/mainPlayer1.png","images/mainPlayer2.png");
  mainRacerImg2= loadAnimation("images/mainPlayer3.png");
  
  oppPink1Img = loadAnimation("images/opponent1.png","images/opponent2.png");
  oppPink2Img = loadAnimation("images/opponent3.png");
  
  oppYellow1Img = loadAnimation("images/opponent4.png","images/opponent5.png");
  oppYellow2Img = loadAnimation("images/opponent6.png");
  
  oppRed1Img = loadAnimation("images/opponent7.png","images/opponent8.png");
  oppRed2Img = loadAnimation("images/opponent9.png");
  
  cycleBell = loadSound("sound/bell.mp3");
  gameOverImg = loadImage("images/gameOver.png");
}

function setup(){
  
createCanvas(1200,300);
// Moving background
path=createSprite(100,150);
path.addImage(pathImg);
path.velocityX = -5;

//creating boy running
mainCyclist  = createSprite(70,150);
mainCyclist.addAnimation("SahilRunning",mainRacerImg1);
mainCyclist.scale=0.07;
  
//set collider for mainCyclist

  
gameOver = createSprite(650,150);
gameOver.addImage(gameOverImg);
gameOver.scale = 0.8;
gameOver.visible = false;  
  
pinkCG = new Group();
yellowCG = new Group();
redCG = new Group();
  
}

function draw() {
  background(0);
  
  drawSprites();
  textSize(20);
  fill(255);
  text("Distance: "+ distance,900,30);
  
  if(gameState===PLAY){
    
   distance = distance + Math.round(getFrameRate()/50);
   path.velocityX = -(6 + 2*distance/150);
  
   mainCyclist.y = World.mouseY;
  
   edges= createEdgeSprites();
   mainCyclist .collide(edges);
  
  //code to reset the background
  if(path.x < 0 ){
    path.x = width/2;
  }
  
    //code to play cycle bell sound
  if(keyDown("space")) {
    cycleBell.play();
  }
  
  //creating continous opponent players
  var select_oppPlayer = Math.round(random(1,3));
  
  if (World.frameCount % 150 == 0) {
    if (select_oppPlayer == 1) {
      pinkCyclists();
    } else if (select_oppPlayer == 2) {
      yellowCyclists();
    } else {
      redCyclists();
    }
  }
  
   if(pinkCG.isTouching(mainCyclist)){
     gameState = END;
     player1.velocityY = 0;
     player1.addAnimation("opponentPlayer1",oppPink2Img);
    }
    
    if(yellowCG.isTouching(mainCyclist)){
      gameState = END;
      player2.velocityY = 0;
      player2.addAnimation("opponentPlayer2",oppYellow2Img);
    }
    
    if(redCG.isTouching(mainCyclist)){
      gameState = END;
      player3.velocityY = 0;
      player3.addAnimation("opponentPlayer3",oppRed2Img);
    }
    
}else if (gameState === END) {
    gameOver.visible = true;
    //Add code to show restart game instrution in text here
  
  
    path.velocityX = 0;
    mainCyclist.velocityY = 0;
    mainCyclist.addAnimation("SahilRunning",mainRacerImg2);
  
    pinkCG.setVelocityXEach(0);
    pinkCG.setLifetimeEach(-1);
  
    yellowCG.setVelocityXEach(0);
    yellowCG.setLifetimeEach(-1);
  
    redCG.setVelocityXEach(0);
    redCG.setLifetimeEach(-1);

    //write condition for calling reset( )
}
}

function pinkCyclists(){
        player1 =createSprite(1100,Math.round(random(50, 250)));
        player1.scale =0.06;
        player1.velocityX = -(6 + 2*distance/150);
        player1.addAnimation("opponentPlayer1",oppPink1Img);
        player1.setLifetime=170;
        pinkCG.add(player1);
}

function yellowCyclists(){
        player2 =createSprite(1100,Math.round(random(50, 250)));
        player2.scale =0.06;
        player2.velocityX = -(6 + 2*distance/150);
        player2.addAnimation("opponentPlayer2",oppYellow1Img);
        player2.setLifetime=170;
        yellowCG.add(player2);
}

function redCyclists(){
        player3 =createSprite(1100,Math.round(random(50, 250)));
        player3.scale =0.06;
        player3.velocityX = -(6 + 2*distance/150);
        player3.addAnimation("opponentPlayer3",oppRed1Img);
        player3.setLifetime=170;
        redCG.add(player3);
}

//create reset function here







```